# Student Habits vs Academic Performance — MLflow Experiment Tracking

This notebook trains two regression models on the **Student Habits vs Academic Performance** dataset and tracks every experiment with MLflow.

**Models trained:**
- Experiment 1 — Ridge Regression (linear baseline, multiple `alpha` values)
- Experiment 2 — Random Forest Regressor (ensemble, multiple `n_estimators` values)

**Dataset:** https://www.kaggle.com/datasets/jayaantanaath/student-habits-vs-academic-performance

## 1. Load Data

In [1]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd

# Download the dataset
kagglehub.dataset_download("jayaantanaath/student-habits-vs-academic-performance")

# Load into a DataFrame
df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "jayaantanaath/student-habits-vs-academic-performance",
    "student_habits_performance.csv",
)

df.head()

/var/folders/fm/zzjr3l5d7nl5vdp9q79tzwvh0000gn/T/ipykernel_80443/3252463466.py:9: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


,student_id,age,gender,study_hours_per_day,social_media_hours,netflix_hours,part_time_job,attendance_percentage,sleep_hours,diet_quality,exercise_frequency,parental_education_level,internet_quality,mental_health_rating,extracurricular_participation,exam_score
0,S1000,23,Female,0.0,1.2,1.1,No,85.0,8.0,Fair,6,Master,Average,8,Yes,56.2
1,S1001,20,Female,6.9,2.8,2.3,No,97.3,4.6,Good,6,High School,Average,8,No,100.0
2,S1002,21,Male,1.4,3.1,1.3,No,94.8,8.0,Poor,1,High School,Poor,1,No,34.3
3,S1003,23,Female,1.0,3.9,1.0,No,71.0,9.2,Poor,4,Master,Good,1,Yes,26.8
4,S1004,19,Female,5.0,4.4,0.5,No,90.9,4.9,Fair,3,Master,Good,1,No,66.4


In [2]:
# Dataset shape
print("Shape:", df.shape)

Shape: (1000, 16)


In [3]:
# Is the target balanced?
df['exam_score'].describe()

count    1000.000000
mean       69.601500
std        16.888564
min        18.400000
25%        58.475000
50%        70.500000
75%        81.325000
max       100.000000
Name: exam_score, dtype: float64

## 2. Check for Missing Values

In [4]:
# Check for missing values in each column
missing_values = df.isnull().sum()
print("Missing values per column:")
print(missing_values)

# Total missing values in the dataset
total_missing = df.isnull().sum().sum()
print(f"\nTotal missing values: {total_missing}")

# Summary of data types and non-null counts
df.info()

Missing values per column:
student_id                        0
age                               0
gender                            0
study_hours_per_day               0
social_media_hours                0
netflix_hours                     0
part_time_job                     0
attendance_percentage             0
sleep_hours                       0
diet_quality                      0
exercise_frequency                0
parental_education_level         91
internet_quality                  0
mental_health_rating              0
extracurricular_participation     0
exam_score                        0
dtype: int64

Total missing values: 91
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 16 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   student_id                     1000 non-null   object 
 1   age                            1000 non-null   int64  
 2   gender 

In [5]:
df.describe()

,age,study_hours_per_day,social_media_hours,netflix_hours,attendance_percentage,sleep_hours,exercise_frequency,mental_health_rating,exam_score
count,1000.0000,1000.00000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,20.4980,3.55010,2.505500,1.819700,84.131700,6.470100,3.042000,5.438000,69.601500
std,2.3081,1.46889,1.172422,1.075118,9.399246,1.226377,2.025423,2.847501,16.888564
min,17.0000,0.00000,0.000000,0.000000,56.000000,3.200000,0.000000,1.000000,18.400000
25%,18.7500,2.60000,1.700000,1.000000,78.000000,5.600000,1.000000,3.000000,58.475000
50%,20.0000,3.50000,2.500000,1.800000,84.400000,6.500000,3.000000,5.000000,70.500000
75%,23.0000,4.50000,3.300000,2.525000,91.025000,7.300000,5.000000,8.000000,81.325000
max,24.0000,8.30000,7.200000,5.400000,100.000000,10.000000,6.000000,10.000000,100.000000


## 3. Preprocessing

In [6]:
# Fill missing parental_education_level with the column mode
df['parental_education_level'] = df['parental_education_level'].fillna(
    df['parental_education_level'].mode()[0]
)

# Confirm no more missing values
print("Missing values after fill:")
print(df.isnull().sum())

Missing values after fill:
student_id                       0
age                              0
gender                           0
study_hours_per_day              0
social_media_hours               0
netflix_hours                    0
part_time_job                    0
attendance_percentage            0
sleep_hours                      0
diet_quality                     0
exercise_frequency               0
parental_education_level         0
internet_quality                 0
mental_health_rating             0
extracurricular_participation    0
exam_score                       0
dtype: int64


In [7]:
# Ordinal-encode categorical columns
gender_map = {"Male": 0, "Female": 1, "Other": 2}
diet_map   = {"Poor": 0, "Fair": 1, "Good": 2}
par_ed_map = {"High School": 0, "Bachelor": 1, "Master": 2}
internet_map = {"Poor": 0, "Average": 1, "Good": 2}
clubs_map  = {"No": 0, "Yes": 1}
job_map    = {"No": 0, "Yes": 1}

df["gender"]                       = df["gender"].str.strip().str.title().map(gender_map)
df["diet_quality"]                  = df["diet_quality"].str.strip().str.title().map(diet_map)
df["parental_education_level"]      = df["parental_education_level"].str.strip().str.title().map(par_ed_map)
df["internet_quality"]              = df["internet_quality"].str.strip().str.title().map(internet_map)
df["extracurricular_participation"] = df["extracurricular_participation"].str.strip().str.title().map(clubs_map)
df["part_time_job"]                 = df["part_time_job"].str.strip().str.title().map(job_map)

# Drop the student_id column — it carries no predictive signal
df = df.drop(columns=["student_id"], errors="ignore")

df.head()

,age,gender,study_hours_per_day,social_media_hours,netflix_hours,part_time_job,attendance_percentage,sleep_hours,diet_quality,exercise_frequency,parental_education_level,internet_quality,mental_health_rating,extracurricular_participation,exam_score
0,23,1,0.0,1.2,1.1,0,85.0,8.0,1,6,2,1,8,1,56.2
1,20,1,6.9,2.8,2.3,0,97.3,4.6,2,6,0,1,8,0,100.0
2,21,0,1.4,3.1,1.3,0,94.8,8.0,0,1,0,0,1,0,34.3
3,23,1,1.0,3.9,1.0,0,71.0,9.2,0,4,2,2,1,1,26.8
4,19,1,5.0,4.4,0.5,0,90.9,4.9,1,3,2,2,1,0,66.4


In [8]:
df.columns

Index(['age', 'gender', 'study_hours_per_day', 'social_media_hours',
       'netflix_hours', 'part_time_job', 'attendance_percentage',
       'sleep_hours', 'diet_quality', 'exercise_frequency',
       'parental_education_level', 'internet_quality', 'mental_health_rating',
       'extracurricular_participation', 'exam_score'],
      dtype='object')

## 4. Train / Validation / Test Split

In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Define features (X) and target (y)
TARGET = "exam_score"
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Bin target for stratified splitting (keeps score distributions balanced)
y_binned = pd.cut(y, bins=5, labels=False)

# First split: 60 % train, 40 % temp
X_train, X_temp, y_train, y_temp, yb_train, yb_temp = train_test_split(
    X, y, y_binned, test_size=0.40, random_state=42, stratify=y_binned
)

# Second split: 20 % validation, 20 % test from temp
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=yb_temp
)

print(f"Train set:      {X_train.shape[0]} samples")
print(f"Validation set: {X_val.shape[0]} samples")
print(f"Test set:       {X_test.shape[0]} samples")

Train set:      600 samples
Validation set: 200 samples
Test set:       200 samples


## 5. Scale Features

In [10]:
from sklearn.preprocessing import StandardScaler
import joblib
import pathlib

# Save artefacts next to the notebook using an absolute path,
# so predict_pipeline.py can always find them regardless of cwd.
PROJECT_DIR = pathlib.Path.cwd()
SCALER_PATH = PROJECT_DIR / 'scaler.joblib'

# Fit scaler on training data only
scaler = StandardScaler()
scaler.fit(X_train)

# Save the scaler for use in the prediction pipeline
joblib.dump(scaler, SCALER_PATH)
print(f'Scaler saved to {SCALER_PATH}')

# Transform all splits
X_train_s = scaler.transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)


Scaler saved to /Users/tiana/Desktop/HSLU/Semester2/ARTIFIN/Project/scaler.joblib


## 6. MLflow Setup

In [ ]:
import os
import shutil
import subprocess
import pathlib
import mlflow
import mlflow.sklearn

# Resolve project root via git (correct regardless of where Jupyter was started)
try:
    _git_root = subprocess.check_output(
        ['git', 'rev-parse', '--show-toplevel'], stderr=subprocess.DEVNULL
    ).decode().strip()
    PROJECT_DIR = pathlib.Path(_git_root)
except Exception:
    PROJECT_DIR = pathlib.Path.cwd()

MLRUNS_DIR = PROJECT_DIR / 'mlruns'

# Clean old setup if it exists (optional, keeps things tidy)
old_db = PROJECT_DIR / 'mlflow.db'
old_artifacts = PROJECT_DIR / 'mlartifacts'
if old_db.exists():
    old_db.unlink()
    print('Removed old mlflow.db')
if old_artifacts.exists():
    shutil.rmtree(old_artifacts)
    print('Removed old mlartifacts/')

# Create mlruns directory for file-based backend
MLRUNS_DIR.mkdir(parents=True, exist_ok=True)

# File-based backend store (everything in mlruns/)
MLFLOW_TRACKING_URI = f'file:///{MLRUNS_DIR}'

# Set tracking URI
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

print('Project dir   :', PROJECT_DIR)
print('Tracking URI  :', MLFLOW_TRACKING_URI)
print()
print('To open the MLflow UI run this command from your terminal:')
print(f'  mlflow ui --backend-store-uri {MLFLOW_TRACKING_URI} --port 5000')

Removed old mlflow.db
Removed old mlartifacts/
Project dir   : /Users/tiana/Desktop/HSLU/Semester2/ARTIFIN/Project
Tracking URI  : sqlite:////Users/tiana/Desktop/HSLU/Semester2/ARTIFIN/Project/mlflow.db
Artifact root : file:///Users/tiana/Desktop/HSLU/Semester2/ARTIFIN/Project/mlartifacts

To open the MLflow UI run this command from your terminal:
  mlflow ui --backend-store-uri sqlite:////Users/tiana/Desktop/HSLU/Semester2/ARTIFIN/Project/mlflow.db --default-artifact-root file:///Users/tiana/Desktop/HSLU/Semester2/ARTIFIN/Project/mlartifacts --port 5000


---
## 7. Experiment 1 — Ridge Regression

We sweep over multiple `alpha` regularisation values and log each as a separate MLflow run.

In [ ]:
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Set the experiment by name (MLflow will create it if it doesn't exist)
mlflow.set_experiment('Student_Habits_Ridge_Regression')

2026/03/26 09:51:34 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/03/26 09:51:34 INFO mlflow.store.db.utils: Updating database tables
2026/03/26 09:51:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/26 09:51:35 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'student_ridge_alpha_0.1'.
2026/03/26 09:51:36 WARNING mlflow.tracking._model_registry.fluent: Run with id dfb000e4a9b145a0b5526cbbb7d2ed1b has no artifacts at artifact path 'model', registering model based on models:/m-fef7e6a7e52948b6819f8e2d7a1e9f1e instead
Created version '1' of model 'student

alpha=0.1     val_r2=0.8858  test_rmse=5.0352  run_id=dfb000e4a9b145a0b5526cbbb7d2ed1b


Successfully registered model 'student_ridge_alpha_1.0'.
2026/03/26 09:51:38 WARNING mlflow.tracking._model_registry.fluent: Run with id 24875fd5c2ba43e2afe7741b8dadab49 has no artifacts at artifact path 'model', registering model based on models:/m-931fbedcc7fd42d0a626fba17e32f45e instead
Created version '1' of model 'student_ridge_alpha_1.0'.
2026/03/26 09:51:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/26 09:51:38 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


alpha=1.0     val_r2=0.8857  test_rmse=5.0382  run_id=24875fd5c2ba43e2afe7741b8dadab49


Successfully registered model 'student_ridge_alpha_10.0'.
2026/03/26 09:51:39 WARNING mlflow.tracking._model_registry.fluent: Run with id 26384c386f8a4ce4b4de11b3ee6859c3 has no artifacts at artifact path 'model', registering model based on models:/m-5175d22591884761b080f6a80cb42cca instead
Created version '1' of model 'student_ridge_alpha_10.0'.
2026/03/26 09:51:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/26 09:51:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


alpha=10.0    val_r2=0.8846  test_rmse=5.0723  run_id=26384c386f8a4ce4b4de11b3ee6859c3


Successfully registered model 'student_ridge_alpha_100.0'.
2026/03/26 09:51:40 WARNING mlflow.tracking._model_registry.fluent: Run with id d98193438b1241b2bbb2a86f3e41b777 has no artifacts at artifact path 'model', registering model based on models:/m-9fed80917dd84cad94d71f8ad14dada6 instead


alpha=100.0   val_r2=0.8598  test_rmse=5.7122  run_id=d98193438b1241b2bbb2a86f3e41b777


Created version '1' of model 'student_ridge_alpha_100.0'.


### Find the best Ridge run

In [13]:
# Find the best Ridge run (lowest test RMSE)
ridge_runs = mlflow.search_runs(
    experiment_names=["Student_Habits_Ridge_Regression"],
    order_by=["metrics.test_rmse ASC"]
)

if ridge_runs.empty:
    # Debug: show what experiments actually exist
    client = mlflow.tracking.MlflowClient()
    existing = [e.name for e in client.search_experiments()]
    raise RuntimeError(
        f"No Ridge runs found. Existing experiments: {existing}\n"
        "Make sure the Ridge training cell ran successfully before this cell."
    )

best_ridge_run = ridge_runs.iloc[0]
best_ridge_run_id    = best_ridge_run["run_id"]
best_ridge_alpha     = best_ridge_run["params.alpha"]
best_ridge_test_rmse = best_ridge_run["metrics.test_rmse"]
best_ridge_test_r2   = best_ridge_run["metrics.test_r2"]

print(f"Best Ridge Model:")
print(f"  alpha     = {best_ridge_alpha}")
print(f"  test_rmse = {best_ridge_test_rmse:.4f}")
print(f"  test_r2   = {best_ridge_test_r2:.4f}")

# Register best Ridge model
mlflow.register_model(f"runs:/{best_ridge_run_id}/model", "best_student_ridge")
print("\nBest Ridge model registered as 'best_student_ridge'")


Successfully registered model 'best_student_ridge'.
2026/03/26 09:51:40 WARNING mlflow.tracking._model_registry.fluent: Run with id dfb000e4a9b145a0b5526cbbb7d2ed1b has no artifacts at artifact path 'model', registering model based on models:/m-fef7e6a7e52948b6819f8e2d7a1e9f1e instead


Best Ridge Model:
  alpha     = 0.1
  test_rmse = 5.0352
  test_r2   = 0.9024

Best Ridge model registered as 'best_student_ridge'


Created version '1' of model 'best_student_ridge'.


---
## 8. Experiment 2 — Random Forest Regressor

We sweep over multiple `n_estimators` values to find the best ensemble size.

In [14]:
from sklearn.ensemble import RandomForestRegressor

# Create experiment only if it doesn't already exist with the right artifact root
EXP_RF = 'Student_Habits_Random_Forest'
if mlflow.get_experiment_by_name(EXP_RF) is None:
    mlflow.create_experiment(EXP_RF, artifact_location=MLFLOW_ARTIFACT_ROOT)
mlflow.set_experiment(EXP_RF)

n_estimators_list = [50, 100, 200, 300]

for n_est in n_estimators_list:
    with mlflow.start_run(run_name=f'RF_n_estimators_{n_est}'):

        # --- Train ---
        model = RandomForestRegressor(
            n_estimators=n_est,
            max_depth=10,
            min_samples_leaf=4,
            random_state=42,
            n_jobs=-1
        )
        model.fit(X_train_s, y_train)

        # --- Predict ---
        y_val_pred  = model.predict(X_val_s)
        y_test_pred = model.predict(X_test_s)

        # --- Metrics ---
        val_rmse  = float(np.sqrt(mean_squared_error(y_val,  y_val_pred)))
        val_mae   = float(mean_absolute_error(y_val,  y_val_pred))
        val_r2    = float(r2_score(y_val,  y_val_pred))

        test_rmse = float(np.sqrt(mean_squared_error(y_test, y_test_pred)))
        test_mae  = float(mean_absolute_error(y_test, y_test_pred))
        test_r2   = float(r2_score(y_test, y_test_pred))

        # --- Log parameters ---
        mlflow.log_param('model_type',       'RandomForest')
        mlflow.log_param('n_estimators',     n_est)
        mlflow.log_param('max_depth',        10)
        mlflow.log_param('min_samples_leaf', 4)
        mlflow.log_param('random_state',     42)

        # --- Log metrics ---
        mlflow.log_metric('val_rmse',  val_rmse)
        mlflow.log_metric('val_mae',   val_mae)
        mlflow.log_metric('val_r2',    val_r2)
        mlflow.log_metric('test_rmse', test_rmse)
        mlflow.log_metric('test_mae',  test_mae)
        mlflow.log_metric('test_r2',   test_r2)

        # --- Log model (artifact_path is the folder name inside the run) ---
        mlflow.sklearn.log_model(sk_model=model, artifact_path='model')

        run_id = mlflow.active_run().info.run_id
        print(f'n_estimators={n_est:<4}  val_r2={val_r2:.4f}  test_rmse={test_rmse:.4f}  run_id={run_id}')

    mlflow.register_model(f'runs:/{run_id}/model', f'student_rf_{n_est}_trees')


2026/03/26 09:51:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/26 09:51:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'student_rf_50_trees'.
2026/03/26 09:51:42 WARNING mlflow.tracking._model_registry.fluent: Run with id b73576c2b0dd41c199841bac9ad72617 has no artifacts at artifact path 'model', registering model based on models:/m-2aa9429bab144558b81e6251740fae1c instead
Created version '1' of model 'student_rf_50_trees'.
2026/03/26 09:51:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/26 09:51:42 WARNING mlflow.sklearn: Sa

n_estimators=50    val_r2=0.8394  test_rmse=6.8091  run_id=b73576c2b0dd41c199841bac9ad72617


Successfully registered model 'student_rf_100_trees'.
2026/03/26 09:51:43 WARNING mlflow.tracking._model_registry.fluent: Run with id 18686f9e233844d8a562e4451be69880 has no artifacts at artifact path 'model', registering model based on models:/m-25ac6f70e12d4a5da9ade929491f89e1 instead
Created version '1' of model 'student_rf_100_trees'.
2026/03/26 09:51:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/26 09:51:44 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


n_estimators=100   val_r2=0.8387  test_rmse=6.7821  run_id=18686f9e233844d8a562e4451be69880


Successfully registered model 'student_rf_200_trees'.
2026/03/26 09:51:45 WARNING mlflow.tracking._model_registry.fluent: Run with id f54672a8af14493f836ae0936d39027e has no artifacts at artifact path 'model', registering model based on models:/m-1498806ad2364dada07139a8a2ab0a11 instead
Created version '1' of model 'student_rf_200_trees'.
2026/03/26 09:51:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


n_estimators=200   val_r2=0.8370  test_rmse=6.8217  run_id=f54672a8af14493f836ae0936d39027e


2026/03/26 09:51:45 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'student_rf_300_trees'.
2026/03/26 09:51:46 WARNING mlflow.tracking._model_registry.fluent: Run with id 8b879fab9bc34abc8bc29391873c0ef0 has no artifacts at artifact path 'model', registering model based on models:/m-e9cfde211219499a85f0eb30b11bc203 instead


n_estimators=300   val_r2=0.8372  test_rmse=6.7892  run_id=8b879fab9bc34abc8bc29391873c0ef0


Created version '1' of model 'student_rf_300_trees'.


### Find the best Random Forest run

In [15]:
# Find the best Random Forest run (lowest test RMSE)
rf_runs = mlflow.search_runs(
    experiment_names=["Student_Habits_Random_Forest"],
    order_by=["metrics.test_rmse ASC"]
)

if rf_runs.empty:
    client = mlflow.tracking.MlflowClient()
    existing = [e.name for e in client.search_experiments()]
    raise RuntimeError(
        f"No Random Forest runs found. Existing experiments: {existing}\n"
        "Make sure the Random Forest training cell ran successfully before this cell."
    )

best_rf_run = rf_runs.iloc[0]
best_rf_run_id    = best_rf_run["run_id"]
best_rf_n_est     = best_rf_run["params.n_estimators"]
best_rf_test_rmse = best_rf_run["metrics.test_rmse"]
best_rf_test_r2   = best_rf_run["metrics.test_r2"]

print(f"Best Random Forest Model:")
print(f"  n_estimators = {best_rf_n_est}")
print(f"  test_rmse    = {best_rf_test_rmse:.4f}")
print(f"  test_r2      = {best_rf_test_r2:.4f}")

# Register best RF model
mlflow.register_model(f"runs:/{best_rf_run_id}/model", "best_student_rf")
print("\nBest Random Forest model registered as 'best_student_rf'")


Successfully registered model 'best_student_rf'.
2026/03/26 09:51:46 WARNING mlflow.tracking._model_registry.fluent: Run with id 18686f9e233844d8a562e4451be69880 has no artifacts at artifact path 'model', registering model based on models:/m-25ac6f70e12d4a5da9ade929491f89e1 instead


Best Random Forest Model:
  n_estimators = 100
  test_rmse    = 6.7821
  test_r2      = 0.8230

Best Random Forest model registered as 'best_student_rf'


Created version '1' of model 'best_student_rf'.


---
## 9. Compare Models & Register the Overall Best

In [16]:
print("=" * 50)
print("Model Comparison — Test Set")
print("=" * 50)
print(f"Ridge Regression   test_rmse={best_ridge_test_rmse:.4f}  test_r2={best_ridge_test_r2:.4f}")
print(f"Random Forest      test_rmse={best_rf_test_rmse:.4f}  test_r2={best_rf_test_r2:.4f}")

if best_rf_test_rmse < best_ridge_test_rmse:
    overall_best_run_id = best_rf_run_id
    overall_best_label  = f"Random Forest (n_estimators={best_rf_n_est})"
else:
    overall_best_run_id = best_ridge_run_id
    overall_best_label  = f"Ridge Regression (alpha={best_ridge_alpha})"

print(f"\n✅ Overall best model: {overall_best_label}")

# Register the overall best model
mlflow.register_model(f"runs:/{overall_best_run_id}/model", "BestStudentHabitsModel")
print("Overall best model registered as 'BestStudentHabitsModel'")

Successfully registered model 'BestStudentHabitsModel'.
2026/03/26 09:51:46 WARNING mlflow.tracking._model_registry.fluent: Run with id dfb000e4a9b145a0b5526cbbb7d2ed1b has no artifacts at artifact path 'model', registering model based on models:/m-fef7e6a7e52948b6819f8e2d7a1e9f1e instead


Model Comparison — Test Set
Ridge Regression   test_rmse=5.0352  test_r2=0.9024
Random Forest      test_rmse=6.7821  test_r2=0.8230

✅ Overall best model: Ridge Regression (alpha=0.1)
Overall best model registered as 'BestStudentHabitsModel'


Created version '1' of model 'BestStudentHabitsModel'.


## 10. Transition Best Model to Staging

In [17]:
from mlflow.tracking import MlflowClient
from datetime import datetime

# Pass the same URI so MlflowClient points at the right database
client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)
date = datetime.today().strftime('%Y-%m-%d')

# Set an alias instead of the deprecated 'Staging' stage
client.set_registered_model_alias(
    name='BestStudentHabitsModel',
    alias='Staging',
    version=1
)
print("Model 'BestStudentHabitsModel' version 1 aliased as 'Staging'")

# Update the model version description
client.update_model_version(
    name='BestStudentHabitsModel',
    version=1,
    description=(
        f'Best model from Student Habits experiments: {overall_best_label}. '
        f'test_rmse={min(best_ridge_test_rmse, best_rf_test_rmse):.4f} as of {date}'
    )
)
print('Model description updated.')


Model 'BestStudentHabitsModel' version 1 aliased as 'Staging'
Model description updated.


## 11. Load Best Model from Registry & Evaluate

In [18]:
import mlflow.pyfunc
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load the best model using the alias set above
model_uri    = 'models:/BestStudentHabitsModel@Staging'
loaded_model = mlflow.pyfunc.load_model(model_uri)

# Make predictions on the test set
predictions = loaded_model.predict(X_test_s)

# Display first 10 predictions vs actuals
print('First 10 predictions vs actual exam scores:')
for pred, actual in zip(predictions[:10], y_test.values[:10]):
    print(f'  predicted={pred:.2f}   actual={actual:.2f}')

# Final evaluation
rmse = float(np.sqrt(mean_squared_error(y_test, predictions)))
mae  = float(mean_absolute_error(y_test, predictions))
r2   = float(r2_score(y_test, predictions))

print(f'\nFinal Test Metrics (loaded from registry):')
print(f'  RMSE : {rmse:.4f}')
print(f'  MAE  : {mae:.4f}')
print(f'  R²   : {r2:.4f}')


First 10 predictions vs actual exam scores:
  predicted=85.02   actual=89.30
  predicted=76.65   actual=76.60
  predicted=73.43   actual=75.30
  predicted=50.53   actual=58.60
  predicted=69.64   actual=62.70
  predicted=66.78   actual=65.20
  predicted=74.34   actual=68.40
  predicted=70.21   actual=72.50
  predicted=76.39   actual=77.60
  predicted=84.53   actual=82.50

Final Test Metrics (loaded from registry):
  RMSE : 5.0352
  MAE  : 4.0712
  R²   : 0.9024


## 12. Save Best Model Locally for FastAPI

The prediction pipeline (`predict_pipeline.py`) loads `best_model.joblib` at startup.

In [19]:
import joblib
import mlflow.sklearn

# Load the sklearn flavour directly so we can call .predict() natively
best_sklearn_model = mlflow.sklearn.load_model(f'runs:/{overall_best_run_id}/model')

# Save to the project directory so predict_pipeline.py always finds it
MODEL_PATH = PROJECT_DIR / 'best_model.joblib'
joblib.dump(best_sklearn_model, MODEL_PATH)
print(f'best_model.joblib saved to {MODEL_PATH}')
print('Ready for predict_pipeline.py')


best_model.joblib saved to /Users/tiana/Desktop/HSLU/Semester2/ARTIFIN/Project/best_model.joblib
Ready for predict_pipeline.py
